# Prompt U-Net Version 333 — Hybrid Separable/Conv2D

## Summary

**V333 uses a hybrid convolution strategy** — SeparableConv2D for most intermediate layers with Conv2D selectively retained at three critical junctions where channel-blind depthwise convolutions would hurt:

| Junction | Conv Type | Reason |
|----------|-----------|--------|
| Strided downsampling (stride=2) | **Conv2D** | Channel mixing essential before spatial reduction |
| Prompt fusion (image branch before Add) | **Conv2D** | Critical handshake between the two encoders |
| Bottleneck (4×4×384) | **Conv2D** | Maximum abstraction, full expressivity worth the cost |
| All other intermediate convs | SeparableConv2D | Parameter-efficient, SE attention compensates |

Only trained on 3600 instead of 4000 epochs.

All other choices identical to v332:

| Component | Choice | Source |
|-----------|--------|--------|
| Architecture | `prompt_unet_333.py` (Hybrid Conv2D/SeparableConv2D + SE Attention) | new |
| SE Attention | enabled (unchanged) | v313 / v332 |
| Offset | 16 | v316 |
| Loss | `binary_crossentropy` (plain BCE) | v316 |
| Training buffer | 10,000 data points | v330 / v331 |
| Refresh cadence | every **30** epochs | v332 |
| Datasets | NAKO + TotalSeg + MSD + BraTS-GLI + BraTS-MEN-RT + TopCoW MR + TopCoW CT | v319 |
| Total patients | 208 | v319 |

**Expected parameter count: ~10-12 M** (vs 28.0 M for v332, ~4-5 M for pure separable v314).

> This isolates: does a hybrid conv strategy retain v332-quality segmentation while cutting parameters by ~60%?

## Setup

In [ ]:
import os
import sys
import gc
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import mlflow
import tensorflow as tf

import logging
tf.get_logger().setLevel(logging.ERROR)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

print(f"TF  : {tf.__version__}")
print(f"GPUs: {tf.config.list_physical_devices('GPU')}")

## Imports

In [ ]:
# Allow importing from project root
notebook_dir = Path().resolve()
project_root  = notebook_dir.parent
sys.path.insert(0, str(project_root))

from data.DataLoader_npz import DataLoader_npz
from data.DataGenerator  import DataGenerator

from utils.augmentations  import PromptUNetAugmenter
from utils.metrics        import dice_score_tf
from utils.visualization  import plot_result

from training.prompt_unet_333 import PromptUNet   # v333 architecture (SeparableConv2D + SE Attention)
from training.optimizer       import PromptUNetOptimizer

## Data Loading

208 patients total (v316 base + TopCoW from v319).

In [ ]:
dataset_paths = [
    "data/train_data/nako_combined.npz",       # 61 PIDs
    "data/train_data/total_seg_combined.npz",  # 45 PIDs
    "data/train_data/msd_combined.npz",        # 40 PIDs
    "data/train_data/brats_gli.npz",           # 20 PIDs
    "data/train_data/brats_men_rt.npz",        #  6 PIDs
    "data/train_data/TopCoW_MR.npz",           # 18 PIDs  (from v319)
    "data/train_data/TopCoW_CT.npz",           # 18 PIDs  (from v319)
]  # Total: 208 patients

dataloader    = DataLoader_npz(dataset_paths, val_size=0.01)
datagenerator = DataGenerator(dataloader)

print(f"Image size: {datagenerator.height} x {datagenerator.width}")

## Hyperparameters

In [ ]:
version           = "p_unet_333"

epochs            = 3600
batch_size        = 128
dp_training       = 10000  # 10 k points per buffer refresh (from v330)
dp_testing        = 1000

offset            = 16     # slice-distance offset (from v316)
max_number_labels = 4

new_ds       = 30    # refresh training data every N epochs
new_val_loop = 300   # run validation every N epochs

# LR schedule phases
warmup_epochs = 50
flat_epochs   = 1500

## Model & Optimizer

`WarmupFlatCosineDecay` schedule (from `optimizer.py`):
- Phase 1 (50 ep): linear warmup 1e-6 → 1e-3  
- Phase 2 (1500 ep): flat plateau at 1e-3  
- Phase 3 (2450 ep): cosine decay 1e-3 → 1e-5

Loss: **plain `binary_crossentropy`**.

**V333 hybrid strategy:** SeparableConv2D everywhere except three critical junctions kept as Conv2D:
1. Strided downsampling (stride=2, channels→2×)
2. Prompt fusion (image branch conv before Add with SE-gated skip)
3. Bottleneck

Input projections (image 1→48, prompt 2→48) and the final 1×1 output layer remain as regular `Conv2D`.

In [ ]:
# Build model
model = PromptUNet(height=datagenerator.height, width=datagenerator.width)
# Loss stays as default binary_crossentropy (set inside PromptUNet.__init__)

# Warm-up forward pass to fully initialise all layers
_dummy_x = tf.random.uniform([1, datagenerator.height, datagenerator.width, 1])
_dummy_p = tf.random.uniform([1, datagenerator.height, datagenerator.width, 2])
_ = model.this([_dummy_x, _dummy_p])

print(f"Trainable params: {model.this.count_params():,}")

# Build optimizer (imported from optimizer.py)
opt_builder = PromptUNetOptimizer(
    epochs        = epochs,
    batch_size    = batch_size,
    dp_training   = dp_training,
    warmup_epochs = warmup_epochs,
    flat_epochs   = flat_epochs,
)
model.optimizer   = opt_builder.get_optimizer()
steps_per_epoch   = opt_builder.steps_per_epoch

## Augmentation Pipeline

Same probabilities as v313–v332.

In [ ]:
augmenter = PromptUNetAugmenter(
    prob_photo             = 0.45,
    prob_gamma             = 0.35,
    prob_noise             = 0.40,
    prob_independent_noise = 0.50,
    prob_geometric         = 0.50,
    prob_morph             = 0.30,
    prob_dropout           = 0.40,
    prob_false_pos         = 0.60,
    gamma_range                 = (0.85, 1.25),
    noise_std_range             = (0.0, 0.10),
    independent_noise_std_range = (0.0, 0.01),
)

## Persistent tf.data Pipeline

The pipeline graph (including `.map(augmenter)`) is built **once** here.  
When fresh training data is needed, only the numpy buffer is swapped — no TF graph nodes accumulate over time.

In [ ]:
# Shared numpy buffer
_buf = {"x": None, "y": None, "p": None, "m": None}

def refresh_train_data():
    """Pull fresh random training data into the numpy buffer."""
    x_np, y_np, p_np, m_np, _ = datagenerator.get_data_points_numpy(
        max_data_points   = dp_training,
        offset            = offset,
        max_number_labels = max_number_labels,
    )
    _buf["x"] = x_np
    _buf["y"] = y_np
    _buf["p"] = p_np
    _buf["m"] = m_np
    gc.collect()


def _data_gen():
    """Yields one shuffled sample at a time from the numpy buffer."""
    n       = len(_buf["x"])
    indices = np.random.permutation(n)
    for i in indices:
        yield _buf["x"][i], _buf["y"][i], _buf["p"][i], _buf["m"][i]


H, W = datagenerator.height, datagenerator.width

# Build the pipeline graph ONCE for the entire training run
train_ds = (
    tf.data.Dataset.from_generator(
        _data_gen,
        output_signature=(
            tf.TensorSpec(shape=(H, W, 1), dtype=tf.float32),  # image
            tf.TensorSpec(shape=(H, W, 1), dtype=tf.float32),  # label
            tf.TensorSpec(shape=(H, W, 2), dtype=tf.float32),  # prompt
            tf.TensorSpec(shape=(),        dtype=tf.float32),  # modality
        )
    )
    .map(augmenter, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size, drop_remainder=True)
    .prefetch(tf.data.AUTOTUNE)
)

print("Pipeline ready.")

## Training

In [ ]:
def fit(epochs):
    mlflow.set_experiment(version)

    with mlflow.start_run():

        mlflow.log_params({
            "batch_size"        : batch_size,
            "max_number_labels" : max_number_labels,
            "num_epochs"        : epochs,
            "dp_training"       : dp_training,
            "offset"            : offset,
            "loss_function"     : "binary_crossentropy",
            "new_ds"            : new_ds,
            "warmup_epochs"     : warmup_epochs,
            "flat_epochs"       : flat_epochs,
            "prob_geometric"    : augmenter.prob_geometric,
            "prob_morph"        : augmenter.prob_morph,
            "gamma_range"       : str(augmenter.gamma_range),
            "trainable_params"  : model.this.count_params(),
            "scale_augmentation": "50% crop 128px / 50% crop [128,256]px resized",
            "leakage_fix"       : "crop origin from support label only",
            "se_attention"      : "enabled",
            "conv_type"         : "hybrid_separable_conv2d",
            "mixed_precision"   : "false",
            "datasets"          : "nako+total_seg+msd+brats_gli+brats_men_rt+TopCoW_MR+TopCoW_CT",
        })